# 10 — Extract independent gate hard negatives and supported variants

Notebook 09 showed that the two-stage gate passes movement-like unsupported exercises as supported. This CPU notebook builds a **new training archive** from classes that were not used in the original Workout/Fitness export.

It adds two types of evidence:

- **hard negatives → `unknown`**: movements such as hip thrust, leg raises and Russian twist that resemble supported exercises but are outside the 15-class target;
- **supported variants**: hammer curl, Romanian deadlift and bench-press variants mapped to their existing canonical classes.

The locked GainsByBrains test video and its labels are never downloaded or used here. Run all cells on **CPU**. The default processes 12 videos per source class (144 videos total), which is a useful controlled batch without spending Colab GPU units.


In [ ]:
%pip install -q --upgrade kagglehub mediapipe==0.10.35 opencv-python-headless pandas==2.2.3 tqdm


In [ ]:
from pathlib import Path
import hashlib
import os
import shutil
import urllib.request

import cv2
import kagglehub
import mediapipe as mp
import numpy as np
import pandas as pd
from google.colab import files
from tqdm.auto import tqdm

SEED = 42
TARGET_FPS = 15
MAX_PER_SOURCE_CLASS = 12
OUTPUT_ROOT = Path("/content/calorie_workoutfitness_gate_hardneg_v1")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Runtime: CPU is recommended")
print("MediaPipe:", mp.__version__)
os.system("df -h /content | tail -1")


## Select evidence without touching the locked test

The source dataset is downloaded into the temporary Colab runtime. Twelve clips are selected deterministically across each sorted class list, so rerunning with the same dataset version selects the same files.

The hard-negative list deliberately avoids close variants that the product should accept. Those variants are mapped to an existing supported class instead.


In [ ]:
DATASET_ROOT = Path(kagglehub.dataset_download("hasyimabdillah/workoutfitness-video"))
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".webm"}

HARD_NEGATIVE_CLASSES = [
    "chest fly machine",
    "hip thrust",
    "leg extension",
    "leg raises",
    "russian twist",
    "t bar row",
    "tricep dips",
]

SUPPORTED_VARIANTS = {
    "decline bench press": "bench_press",
    "hammer curl": "bicep_curl",
    "incline bench press": "bench_press",
    "romanian deadlift": "deadlift",
    "tricep Pushdown": "tricep_extension",
}

CLASS_MAP = {name: "unknown" for name in HARD_NEGATIVE_CLASSES}
CLASS_MAP.update(SUPPORTED_VARIANTS)


def spread_sample(items, limit):
    if limit is None or len(items) <= limit:
        return items
    indices = np.linspace(0, len(items) - 1, num=limit, dtype=int)
    return [items[index] for index in sorted(set(indices.tolist()))]


selected = []
for source_name, canonical_name in CLASS_MAP.items():
    class_dir = DATASET_ROOT / source_name
    videos = sorted(
        path for path in class_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
    )
    chosen = spread_sample(videos, MAX_PER_SOURCE_CLASS)
    role = "hard_negative" if canonical_name == "unknown" else "supported_variant"
    selected.extend((path, source_name, canonical_name, role) for path in chosen)
    print(f"{source_name:25s} -> {canonical_name:18s} {role:18s}: {len(chosen)}/{len(videos)}")

expected = sum(min(MAX_PER_SOURCE_CLASS, len([
    path for path in (DATASET_ROOT / name).rglob("*")
    if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
])) for name in CLASS_MAP)
print("Total selected:", len(selected))
assert len(selected) == expected and len(selected) > 0, "Dataset path or class names changed."


In [ ]:
MODEL_PATH = Path("/content/pose_landmarker_full.task")
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_full/float16/latest/pose_landmarker_full.task"
)
if not MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Pose model ready:", MODEL_PATH.exists(), MODEL_PATH)


## Convert videos to motion coordinates

MediaPipe samples 15 frames per second and finds the body joints. We retain 17 joints, move the coordinate origin to the hip center, and divide by torso scale. The TCN later learns motion from these normalized coordinates instead of raw pixels.


In [ ]:
KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
    shoulder_width = np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])
    torso_length = np.linalg.norm(shoulder_center - hip_center)
    scale = max(float(shoulder_width), float(torso_length), 1e-6)
    xyz = (xyz - hip_center) / scale
    return np.concatenate([xyz, frame[:, 3:4]], axis=1).astype(np.float32)


def fill_missing(sequence):
    shape = sequence.shape
    flat = sequence.reshape(shape[0], -1)
    filled = pd.DataFrame(flat).interpolate(limit_direction="both").fillna(0).to_numpy()
    return filled.reshape(shape).astype(np.float32)


def file_fingerprint(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    size = path.stat().st_size
    digest.update(str(size).encode())
    with path.open("rb") as handle:
        digest.update(handle.read(block_size))
        if size > block_size:
            handle.seek(max(0, size - block_size))
            digest.update(handle.read(block_size))
    return digest.hexdigest()


def extract_video(video_path):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        return None, {"status": "open_failed"}

    source_fps = float(capture.get(cv2.CAP_PROP_FPS) or 0) or 30.0
    sample_interval = 1.0 / TARGET_FPS
    next_sample_s = 0.0
    frame_index = 0
    poses, timestamps = [], []
    detected = 0

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )

    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            time_s = frame_index / source_fps
            frame_index += 1
            if time_s + 1e-9 < next_sample_s:
                continue
            next_sample_s += sample_interval

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(image, int(round(time_s * 1000)))
            if result.pose_landmarks:
                points = result.pose_landmarks[0]
                pose = np.asarray([
                    [points[i].x, points[i].y, points[i].z, float(points[i].visibility or 0.0)]
                    for i in KEEP_17
                ], dtype=np.float32)
                poses.append(normalize_pose(pose))
                detected += 1
            else:
                poses.append(np.full((17, 4), np.nan, dtype=np.float32))
            timestamps.append(time_s)

    capture.release()
    if not poses:
        return None, {"status": "no_frames"}

    sequence = fill_missing(np.stack(poses))
    detection_rate = detected / len(sequence)
    return sequence, {
        "status": "ok" if detection_rate >= 0.5 else "low_detection",
        "source_fps": source_fps,
        "sampled_frames": len(sequence),
        "detection_rate": detection_rate,
        "duration_s": timestamps[-1] if timestamps else 0.0,
        "timestamps_s": np.asarray(timestamps, dtype=np.float32),
    }


In [ ]:
MANIFEST_PATH = OUTPUT_ROOT / "manifest.csv"
rows = []

for video_path, source_class, canonical_class, role in tqdm(selected, desc="Extracting poses"):
    relative_path = video_path.relative_to(DATASET_ROOT)
    clip_id = hashlib.sha1(str(relative_path).encode()).hexdigest()[:16]
    landmarks, info = extract_video(video_path)
    output_path = OUTPUT_ROOT / canonical_class / f"{clip_id}.npz"

    if landmarks is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        timestamps_s = info.pop("timestamps_s")
        np.savez_compressed(
            output_path,
            landmarks=landmarks,
            timestamps_s=timestamps_s,
            label=canonical_class,
            source_dataset="workoutfitness-gate-hardneg-v1",
            original_path=str(relative_path),
            target_fps=TARGET_FPS,
        )

    rows.append({
        "clip_id": clip_id,
        "source_dataset": "workoutfitness-gate-hardneg-v1",
        "person_id": "",
        "group_id": f"workoutfitness-gate-hardneg-v1:{clip_id}",
        "source_class": source_class,
        "canonical_class": canonical_class,
        "training_role": role,
        "original_path": str(relative_path),
        "output_path": str(output_path.relative_to(OUTPUT_ROOT)) if landmarks is not None else "",
        "fingerprint": file_fingerprint(video_path),
        **{key: value for key, value in info.items() if key != "timestamps_s"},
    })
    pd.DataFrame(rows).to_csv(MANIFEST_PATH, index=False)

manifest = pd.DataFrame(rows)
display(manifest.tail())


## Audit and download

Low-detection clips remain visible in the manifest, but the training notebook will use only rows with `status == "ok"`. The output ZIP contains landmarks and metadata, not the original videos.


In [ ]:
npz_files = list(OUTPUT_ROOT.rglob("*.npz"))
valid = manifest[manifest["status"] == "ok"]
print("Saved landmark files:", len(npz_files))
print("Valid clips:", len(valid), "/", len(manifest))
print("Status counts:")
print(manifest["status"].value_counts(dropna=False))
print("\nEvidence by source class:")
display(manifest.groupby(["training_role", "source_class", "canonical_class"]).agg(
    clips=("clip_id", "count"),
    valid=("status", lambda values: int((values == "ok").sum())),
    detection_mean=("detection_rate", "mean"),
    detection_min=("detection_rate", "min"),
).round(3))

assert len(npz_files) == manifest["output_path"].ne("").sum(), "Manifest/NPZ count mismatch."
assert len(valid) >= int(0.8 * len(manifest)), "Too many low-detection clips; inspect before training."
assert set(valid["canonical_class"]) >= {"unknown", "bench_press", "bicep_curl", "deadlift", "tricep_extension"}

zip_path = shutil.make_archive(
    "/content/calorie_workoutfitness_gate_hardneg_v1", "zip", root_dir=OUTPUT_ROOT
)
print("\nZIP ready:", zip_path)
print("ZIP size MB:", round(Path(zip_path).stat().st_size / 1024**2, 2))
files.download(zip_path)


## What comes next

Send `calorie_workoutfitness_gate_hardneg_v1.zip` back to the project. The next training notebook will add it to the six existing landmark archives, preserve group-separated splits, retrain the gate and exercise classifier, and choose its threshold using validation data only. GainsByBrains remains a locked diagnostic benchmark.
